In [1]:
import pandas as pd
from sqlalchemy import create_engine
from config import *
from mapping import COLUMN_MAPPING

In [4]:
df = pd.read_csv("../../Dataset/Processed/featured_sales.csv")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Order Month,Order Month Name,Order Quarter,Order Day,Order Weekday,Shipping Days,Profit Margin,Has Discount,High Profit,Order Size
0,1,INDMKB,2020-11-08,2020-11-11,sas,as,asa,asa,asas,asas,...,11,November,4,8,Sunday,3,16.00,No,High,Large
1,2,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,asasa,Consumer,United States,Henderson,...,11,November,4,8,Sunday,3,30.00,No,High,Large
2,3,CA-2020-138688,2020-06-12,2020-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,6,June,2,12,Friday,4,47.00,No,Low,Small
3,4,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,10,October,4,11,Friday,7,-40.00,Yes,Low,Large
4,5,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,10,October,4,11,Friday,7,11.25,Yes,Low,Small


In [5]:
print(df[["Has Discount", "High Profit"]].head(10))
print(df["Has Discount"].unique())

print(df["High Profit"].unique())

  Has Discount High Profit
0           No        High
1           No        High
2           No         Low
3          Yes         Low
4          Yes         Low
5           No        High
6           No         Low
7          Yes        High
8          Yes         Low
9           No        High
['No' 'Yes']
['High' 'Low']


In [6]:
df.rename(columns=COLUMN_MAPPING, inplace=True)

print(df.columns.tolist())

['RowID', 'OrderID', 'OrderDate', 'ShipDate', 'ShipMode', 'CustomerID', 'CustomerName', 'Segment', 'Country', 'City', 'State', 'PostalCode', 'Region', 'ProductID', 'Category', 'SubCategory', 'ProductName', 'Sales', 'Quantity', 'Discount', 'Profit', 'OrderYear', 'OrderMonth', 'OrderMonthName', 'OrderQuarter', 'OrderDay', 'OrderWeekday', 'ShippingDays', 'ProfitMargin', 'HasDiscount', 'HighProfit', 'OrderSize']


In [ ]:
# from sqlalchemy import create_engine

# engine = create_engine(
#     "mssql+pyodbc://MOHAMED-AHMED/ECommerceAnalytics?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
# )

# with engine.connect() as conn:
#     print("Connection Successful!")

Connection Successful!


In [10]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
df["ShipDate"] = pd.to_datetime(df["ShipDate"])

In [11]:
df["HasDiscount"] = (
    df["HasDiscount"]
    .astype(str)
    .str.strip()
    .map({
        "Yes": True,
        "No": False
    })
)

df["HighProfit"] = (
    df["HighProfit"]
    .astype(str)
    .str.strip()
    .map({
        "High": True,
        "Low": False
    })
)


In [12]:
print(df.dtypes)

print(df.isnull().sum())

RowID                      int64
OrderID                   object
OrderDate         datetime64[ns]
ShipDate          datetime64[ns]
ShipMode                  object
CustomerID                object
CustomerName              object
Segment                   object
Country                   object
City                      object
State                     object
PostalCode                object
Region                    object
ProductID                 object
Category                  object
SubCategory               object
ProductName               object
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
OrderYear                  int64
OrderMonth                 int64
OrderMonthName            object
OrderQuarter               int64
OrderDay                   int64
OrderWeekday              object
ShippingDays               int64
ProfitMargin             float64
HasDiscount                 bool
HighProfit

In [13]:
connection_string = (
    f"mssql+pyodbc://{SERVER}/{DATABASE}"
    f"?driver={DRIVER.replace(' ', '+')}"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

with engine.connect() as conn:
    print("Connection Successful!")

Connection Successful!


In [14]:
print(df["HasDiscount"].unique())
print(df["HighProfit"].unique())

print(df.dtypes)

[False  True]
[ True False]
RowID                      int64
OrderID                   object
OrderDate         datetime64[ns]
ShipDate          datetime64[ns]
ShipMode                  object
CustomerID                object
CustomerName              object
Segment                   object
Country                   object
City                      object
State                     object
PostalCode                object
Region                    object
ProductID                 object
Category                  object
SubCategory               object
ProductName               object
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
OrderYear                  int64
OrderMonth                 int64
OrderMonthName            object
OrderQuarter               int64
OrderDay                   int64
OrderWeekday              object
ShippingDays               int64
ProfitMargin             float64
HasDiscount    

In [15]:
print("=" * 50)
print("Data Validation Summary")
print("=" * 50)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

Data Validation Summary
Rows: 9994
Columns: 32

Missing Values:
0

Duplicate Rows:
0

Data Types:
RowID                      int64
OrderID                   object
OrderDate         datetime64[ns]
ShipDate          datetime64[ns]
ShipMode                  object
CustomerID                object
CustomerName              object
Segment                   object
Country                   object
City                      object
State                     object
PostalCode                object
Region                    object
ProductID                 object
Category                  object
SubCategory               object
ProductName               object
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
OrderYear                  int64
OrderMonth                 int64
OrderMonthName            object
OrderQuarter               int64
OrderDay                   int64
OrderWeekday              object
ShippingDay

In [16]:
print(df.shape)

(9994, 32)


In [21]:
dim_customer = (
    df[
        [
            "CustomerID",
            "CustomerName",
            "Segment"
        ]
    ]
    .drop_duplicates(subset=["CustomerID"], keep="first") # التعديل هنا
    .reset_index(drop=True)
)

In [22]:
print(dim_customer.shape)

print(dim_customer["CustomerID"].duplicated().sum())

(794, 3)
0


In [23]:
dim_customer.to_sql(
    "DimCustomer",
    engine,
    if_exists="append",
    index=False
)

print("DimCustomer Loaded Successfully")

DimCustomer Loaded Successfully


In [24]:
# ==========================================
# Build Product Dimension
# ==========================================

dim_product = (
    df[
        [
            "ProductID",
            "ProductName",
            "Category",
            "SubCategory"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_product.head()

,ProductID,ProductName,Category,SubCategory
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage


In [30]:
dim_product = (
    df[
        [
            "ProductID",
            "ProductName",
            "Category",
            "SubCategory"
        ]
    ]
    .drop_duplicates(subset=["ProductID"], keep="first")
    .reset_index(drop=True)
)

In [31]:
print(dim_product.shape)

print(dim_product["ProductID"].duplicated().sum())

(1862, 4)
0


In [32]:
dim_product.to_sql(
    "DimProduct",
    engine,
    if_exists="append",
    index=False
)

print("DimProduct Loaded Successfully")

DimProduct Loaded Successfully


In [33]:
# ==========================================
# Build Location Dimension
# ==========================================

dim_location = (
    df[
        [
            "Country",
            "Region",
            "State",
            "City",
            "PostalCode"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_location.head()

,Country,Region,State,City,PostalCode
0,asas,INDMKB,asas,asas,42420.0
1,United States,South,Kentucky,Henderson,42420.0
2,United States,West,California,Los Angeles,90036.0
3,United States,South,Florida,Fort Lauderdale,33311.0
4,United States,West,California,Los Angeles,90032.0


In [34]:
print(dim_location.shape)

print(dim_location.duplicated().sum())

(633, 5)
0


In [35]:
dim_location.to_sql(
    "DimLocation",
    engine,
    if_exists="append",
    index=False
)

print("DimLocation Loaded Successfully")

DimLocation Loaded Successfully


In [36]:
# ==========================================
# Build Ship Mode Dimension
# ==========================================

dim_shipmode = (
    df[
        [
            "ShipMode"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_shipmode.head()

,ShipMode
0,sas
1,Second Class
2,Standard Class
3,First Class
4,Same Day


In [37]:
print(dim_shipmode.shape)

(5, 1)


In [38]:
dim_shipmode.to_sql(
    "DimShipMode",
    engine,
    if_exists="append",
    index=False
)

print("DimShipMode Loaded Successfully")

DimShipMode Loaded Successfully


In [39]:
# ==========================================
# Build Date Dimension
# ==========================================

dim_date = (
    df[
        [
            "OrderDate",
            "OrderYear",
            "OrderMonth",
            "OrderMonthName",
            "OrderQuarter",
            "OrderDay",
            "OrderWeekday"
        ]
    ]
    .drop_duplicates()
    .sort_values("OrderDate")
    .reset_index(drop=True)
)

In [40]:
dim_date["DateKey"] = dim_date["OrderDate"].dt.strftime("%Y%m%d").astype(int)

In [41]:
dim_date = dim_date[
    [
        "DateKey",
        "OrderDate",
        "OrderYear",
        "OrderMonth",
        "OrderMonthName",
        "OrderQuarter",
        "OrderDay",
        "OrderWeekday"
    ]
]

dim_date.head()

,DateKey,OrderDate,OrderYear,OrderMonth,OrderMonthName,OrderQuarter,OrderDay,OrderWeekday
0,20180103,2018-01-03,2018,1,January,1,3,Wednesday
1,20180104,2018-01-04,2018,1,January,1,4,Thursday
2,20180105,2018-01-05,2018,1,January,1,5,Friday
3,20180106,2018-01-06,2018,1,January,1,6,Saturday
4,20180107,2018-01-07,2018,1,January,1,7,Sunday


In [42]:
print(dim_date.shape)

print(dim_date["DateKey"].duplicated().sum())

(1236, 8)
0


In [43]:
dim_date.to_sql(
    "DimDate",
    engine,
    if_exists="append",
    index=False
)

print("DimDate Loaded Successfully")

DimDate Loaded Successfully


In [44]:
# ==========================================
# Read Dimension Tables from SQL Server
# ==========================================

customer_dim = pd.read_sql("SELECT * FROM DimCustomer", engine)

product_dim = pd.read_sql("SELECT * FROM DimProduct", engine)

location_dim = pd.read_sql("SELECT * FROM DimLocation", engine)

shipmode_dim = pd.read_sql("SELECT * FROM DimShipMode", engine)

date_dim = pd.read_sql("SELECT * FROM DimDate", engine)

In [45]:
customer_dim.head()

product_dim.head()

location_dim.head()

shipmode_dim.head()

date_dim.head()

,DateKey,OrderDate,OrderYear,OrderMonth,OrderMonthName,OrderQuarter,OrderDay,OrderWeekday
0,20180103,2018-01-03,2018,1,January,1,3,Wednesday
1,20180104,2018-01-04,2018,1,January,1,4,Thursday
2,20180105,2018-01-05,2018,1,January,1,5,Friday
3,20180106,2018-01-06,2018,1,January,1,6,Saturday
4,20180107,2018-01-07,2018,1,January,1,7,Sunday


In [46]:
fact_sales = df.copy()

In [47]:
fact_sales = fact_sales.merge(
    customer_dim[
        [
            "CustomerKey",
            "CustomerID"
        ]
    ],
    on="CustomerID",
    how="left"
)

In [48]:
fact_sales[
    [
        "CustomerID",
        "CustomerKey"
    ]
].head()

,CustomerID,CustomerKey
0,as,1
1,CG-12520,2
2,DV-13045,3
3,SO-20335,4
4,SO-20335,4


In [49]:
fact_sales = fact_sales.merge(
    product_dim[
        [
            "ProductKey",
            "ProductID"
        ]
    ],
    on="ProductID",
    how="left"
)

In [50]:
fact_sales = fact_sales.merge(
    location_dim[
        [
            "LocationKey",
            "Country",
            "Region",
            "State",
            "City",
            "PostalCode"
        ]
    ],
    on=[
        "Country",
        "Region",
        "State",
        "City",
        "PostalCode"
    ],
    how="left"
)

In [51]:
fact_sales = fact_sales.merge(
    shipmode_dim,
    on="ShipMode",
    how="left"
)

In [52]:
fact_sales["DateKey"] = (
    fact_sales["OrderDate"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [53]:
fact_sales = fact_sales.merge(
    date_dim[
        [
            "DateKey"
        ]
    ],
    on="DateKey",
    how="left"
)

In [54]:
fact_sales[
    [
        "CustomerKey",
        "ProductKey",
        "LocationKey",
        "ShipModeKey",
        "DateKey"
    ]
].isnull().sum()

CustomerKey    0
ProductKey     0
LocationKey    0
ShipModeKey    0
DateKey        0
dtype: int64

In [55]:
fact_sales = fact_sales[
    [
        "CustomerKey",
        "ProductKey",
        "LocationKey",
        "ShipModeKey",
        "DateKey",
        "Sales",
        "Quantity",
        "Discount",
        "Profit",
        "ShippingDays",
        "ProfitMargin",
        "HasDiscount",
        "HighProfit",
        "OrderSize"
    ]
]

In [56]:
print(fact_sales.head())

print(fact_sales.shape)

print(fact_sales.isnull().sum())

   CustomerKey  ProductKey  LocationKey  ShipModeKey   DateKey     Sales  \
0            1           1            1            1  20201108  261.9600   
1            2           2            2            2  20201108  731.9400   
2            3           3            3            2  20200612   14.6200   
3            4           4            4            3  20191011  957.5775   
4            4           5            4            3  20191011   22.3680   

   Quantity  Discount    Profit  ShippingDays  ProfitMargin  HasDiscount  \
0         2      0.00   41.9136             3         16.00        False   
1         3      0.00  219.5820             3         30.00        False   
2         2      0.00    6.8714             4         47.00        False   
3         5      0.45 -383.0310             7        -40.00         True   
4         2      0.20    2.5164             7         11.25         True   

   HighProfit OrderSize  
0        True     Large  
1        True     Large  
2       

In [59]:
# ==========================================
# Load FactSales
# ==========================================

fact_sales.to_sql(
    "FactSales",
    engine,
    if_exists="append",
    index=False
)

print("FactSales Loaded Successfully")

FactSales Loaded Successfully
